# PyTorch RNN Example

From [Pytorch RNN example](https://www.youtube.com/watch?v=Gl2WXLIMvKA&list=PLhhyoLH6IjfxeoooqP9rhU3HJIAVAJ3Vz&index=5)

In [3]:
# Imports
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision.datasets as datasets
import torchvision.transforms as transforms

# Set Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Shape of MNIST = Nx1x28x28

# Hyperparameters
input_size = 28
sequence_length = 28
num_layers = 2
hidden_size = 256
num_classes = 10
learning_rate = 0.001
batch_size = 64
num_epochs = 2


In [ ]:
# Create an RNN
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(RNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size*sequence_length, num_classes) # uses info from every hidden state by concat
        
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)
        
        # Forward prop
        out, _ = self.rnn(x, h0) # out contains hidden state at each time step, hn contains final hidden state foe each RNN layer
        out = out.reshape(out.shape[0], -1) # concat sequences with hidden size
        out = self.fc(out)
        return out

In [6]:
model = RNN(input_size, hidden_size, num_layers, num_classes)
model

RNN(
  (rnn): RNN(28, 256, num_layers=2, batch_first=True)
  (fc): Linear(in_features=7168, out_features=10, bias=True)
)

In [9]:

# Load Data
train_dataset = datasets.MNIST(
    root='dataset/',                    # dataset for saving the data
    train=True,                         # Use the training set?
    transform=transforms.ToTensor(),    # Transform the data to tensors
    download=True                       # download if not present in the `dataset/` directory
    )

test_dataset = datasets.MNIST(
    root='dataset/',                    # dataset for saving the data
    train=False,                         # Use the training set?
    transform=transforms.ToTensor(),    # Transform the data to tensors
    download=True                       # download if not present in the `dataset/` directory
    )


train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=True)

# Initialize Network
model = RNN(input_size, hidden_size, num_layers, num_classes).to(device)

# Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Check accuracy on training and test
def check_accuracy(loader, model, epoch):
    if loader.dataset.train:
        print("Testing accuracy on training data")
    
    num_correct = 0
    num_samples = 0
    model.eval()
    
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device).squeeze(1)
            y = y.to(device=device)
            
            scores = model(x)
            
            # 64x10
            _, predictions = scores.max(1) # index of max value
            num_correct += (predictions == y).sum()   # Number of correct predictions
            num_samples += predictions.size(0)        # Number of samples
            
        print(f"Epoch {epoch+1}: {num_correct:,} / {num_samples:,} | Accuracy {float(num_correct)/float(num_samples)*100:.2f}")
            

# Train Network
for epoch in range(num_epochs):
    model.train()
    for batch_idx, (data, targets) in enumerate(train_loader):
        data = data.to(device=device).squeeze(1)
        targets = targets.to(device=device)
                
        # Forward
        scores = model(data)
        loss = criterion(scores, targets)
        
        # Backward
        optimizer.zero_grad() # Set all the gradients to 0 for each batch
        loss.backward()
        
        # Gradient descent or adam step
        optimizer.step()
         
    check_accuracy(test_loader, model, epoch)


Epoch 1: 9,603 / 10,000 | Accuracy 96.03
Epoch 2: 9,684 / 10,000 | Accuracy 96.84
